# 2.f) Build Terrain-Slope Suitability — GAEZ v4 Table A6-8 (seule source numerique existante)

Calcule la note de suitability terrain-pente (S1/S2/N, puis convertie en echelle ordinale 0-4) pour
chaque culture x regime, a la fois sur **ERA5** (reference) et sur les **scenarios climatiques AMOC**
(EC-Earth direct + 20CR-solution, hosing/nohosing), prete a etre utilisee au Module V (le mecanisme
reel — degrader la classe de suitability finale — y sera implemente, pas ici).

**Sources** :
- **Methodologie confirmee v5** (formule de l'indice de Fournier, logique, voir
  `0_reference/wiki_raw/06_Module_IV_Agro-edaphic_suitability.pdf`).
- **Table de notation (A6-8.1 rainfed/sprinkler/drip, A6-8.2 gravity irrigation) confirmee v4
  uniquement** — aucun Appendix 6.x n'a jamais ete republie pour v5 (verifie via le sommaire officiel
  du wiki). Table extraite et verifiee (3 echantillons controles visuellement contre le PDF source) dans
  `0_reference/appendix_tables/terrain_slope_ratings_A6-8_parsed.csv` — voir aussi
  `Appendix_6-8_Terrain_Slope_Ratings_GAEZ_V4.pdf` pour le document de reference complet.
- **Incoherence connue dans la source officielle** : la "Table 6-14" (exemple illustratif, texte
  principal du PDF GAEZ v4) differe legerement de la vraie "Table A6-8.1" (meme cellule
  High/Annuals1/FM<1300 : S2/N vs N a la classe de pente 16-30%). La table A6-8 (l'appendice, source
  systematique) est retenue comme reference.

**Portee** : calcule la note terrain, ne l'applique a rien — comme convenu, cette etape s'arrete la en
attendant le Module V (climat+sol -> classe finale -> degradation terrain).


## 1. Configuration

In [1]:
import os; os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import rasterio
from rasterio.windows import Window
from rasterio.transform import rowcol

BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "0_Input_data").is_dir())  # racine du projet (local ou BDAP)
import sys; sys.path.insert(0, str(BASE)); import crop_utils as cu   # helpers + config reader (project root)
CFG = cu.load_config()   # config/config_<profile>.json, written by 00_config_and_inventory.ipynb
SLOPE_TIF = cu.data_file(CFG, "gaez_slope")  # data path: see 00_config_and_inventory.ipynb
TERRAIN_CSV = BASE / "1_SVM" / "2_soil_suitability_gaez" / "GAEZ_soil_ref" / "appendix_tables" / "terrain_slope_ratings_A6-8_parsed.csv"
OUT_DIR = BASE / "1_SVM" / "2_soil_suitability_gaez" / "terrain_suitability_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

ERA5_PR = cu.data_file(CFG, "era5", "pr")  # data path: see 00_config_and_inventory.ipynb

# Chemins bias-ajustes AMOC (externes au repo, deja utilises par 1_SVM) - EC-Earth direct et 20CR-solution
BA_ECEARTH = cu.data_dir(CFG, "amoc_ba_ecearth")  # data path: see 00_config_and_inventory.ipynb
BA_20CR    = cu.data_dir(CFG, "amoc_ba_20cr")  # data path: see 00_config_and_inventory.ipynb

# Scenarios "voulus" - tolerant si pas encore tous disponibles (ex. 2050 pas encore livre)
SCENARIO_TAGS = ['2025_hosing', '2025_nohosing', '2050_hosing', '2050_nohosing']
AMOC_SOURCES = {
    'ECEarth':      {'dir': BA_ECEARTH, 'pattern': 'pr_adj_{tag}.nc'},
    '20CRsolution': {'dir': BA_20CR,    'pattern': 'pr_adj_{tag}_combined.nc'},
}

CROPS = ['Wheat', 'Maize', 'Potato', 'Rice', 'Soybean']
CROP_TO_GROUP = {'Wheat': 'Annuals 1', 'Maize': 'Annuals 2', 'Potato': 'Annuals 2',
                 'Soybean': 'Annuals 2', 'Rice': 'Annuals 3'}
CROP_NAME_ALIASES = {'Wheat': ['Wheat-spring', 'Wheat-winter'], 'Maize': ['Maize'], 'Potato': ['Potato'],
                      'Rice': ['Rice'], 'Soybean': ['Soybean']}
REGIME_TO_WATER_SYSTEM = {'rf': 'rainfed_sprinkler_drip', 'ir': 'gravity_irrigation'}
INPUT_LEVEL = 'Intermediate'  # production (comme pour le sol) - Low/High dispo dans la table si besoin

for p, label in [(SLOPE_TIF, 'SLOPE-MED'), (TERRAIN_CSV, 'table terrain parsee'), (ERA5_PR, 'ERA5 pr')]:
    print(('OK  ' if p.exists() else 'MISSING '), label, '->', p)
for name, src in AMOC_SOURCES.items():
    print(('OK  ' if src['dir'].exists() else 'MISSING '), name, 'dir ->', src['dir'])


OK   SLOPE-MED -> C:\Users\elisa\OneDrive - Politecnico di Milano\PHD_codes\crop_suitability_code_bdap\0_Input_data\terrain\SLOPE-MED.tif
OK   table terrain parsee -> C:\Users\elisa\OneDrive - Politecnico di Milano\PHD_codes\crop_suitability_code_bdap\1_SVM\2_soil_suitability_gaez\GAEZ_soil_ref\appendix_tables\terrain_slope_ratings_A6-8_parsed.csv
OK   ERA5 pr -> C:\Users\elisa\OneDrive - Politecnico di Milano\PHD_codes\crop_suitability_code_bdap\0_Input_data\ERA5 data\pr_ERA5_1981_2015_daily_025deg.nc
OK   ECEarth dir -> C:\Users\elisa\OneDrive - Politecnico di Milano\PHD_codes\bias adjustment_codes\5)ECearth\3)BA\outputs_BA\1deg
OK   20CRsolution dir -> C:\Users\elisa\OneDrive - Politecnico di Milano\PHD_codes\bias adjustment_codes\1) 20CR solution\4) bias_adjustment\outputs_BA\1deg


## 2. Indice de Fournier modifie et classes

`Fm = 12 . sum(Pi^2) / sum(Pi)` sur les 12 mois (Pi = precipitation mensuelle). Calcule a la
resolution native de chaque source de precipitation (0.25 deg ERA5, 1 deg AMOC) — pas de
reechantillonnage avant calcul, pour rester fidele a chaque source.

In [2]:
FM_BREAKS = [1300, 1800, 2200, 2500, 2700]  # bornes -> 6 classes
FM_LABELS = ['<1300', '1300-1800', '1800-2200', '2200-2500', '2500-2700', '>2700']


def compute_fournier(pr_daily, time_dim='time'):
    """pr_daily : DataArray (time, ...) en mm/jour. Retourne Fm (sans dim temps) et sa classe (str)."""
    monthly = pr_daily.resample({time_dim: '1MS'}).sum()  # somme mensuelle (mm/mois)
    # moyenne sur toutes les annees disponibles, par mois calendaire (climatologie)
    monthly_clim = monthly.groupby(f'{time_dim}.month').mean(time_dim)  # dim 'month' = 1..12
    sum_p = monthly_clim.sum('month')
    sum_p2 = (monthly_clim ** 2).sum('month')
    with np.errstate(invalid='ignore', divide='ignore'):
        fm = 12.0 * sum_p2 / sum_p
    return fm


def classify_fournier(fm):
    idx = xr.zeros_like(fm, dtype=np.int8)
    for b in FM_BREAKS:
        idx = idx + (fm >= b).astype(np.int8)
    return idx  # 0..5 -> index dans FM_LABELS


## 3. Classes de pente (SLOPE-MED -> 8 classes A6-8)

In [3]:
# Legende SLOPE-MED confirmee empiriquement (zone plate vs Alpes vs mer) :
# 0 = eau/sans donnee ; 1..10 = les 10 classes officielles, de 0-0.5% (1) a >45% (10)
SLOPEMED_10_TO_A68_8 = {
    1: '0-0.5', 2: '0.5-2', 3: '2-5', 4: '5-8',
    5: '8-16', 6: '8-16',      # 8-12 et 12-16 fusionnes (A6-8 n'a que 8-16)
    7: '16-30', 8: '16-30',    # 16-24 et 24-30 fusionnes
    9: '30-45', 10: '>45',
}
SLOPE_CLASSES_A68 = ['0-0.5', '0.5-2', '2-5', '5-8', '8-16', '16-30', '30-45', '>45']
SLOPE_LABEL_TO_IDX = {s: i for i, s in enumerate(SLOPE_CLASSES_A68)}

# table de reclassification directe code brut (0-10) -> indice A6-8 (0-7), -1 = pas de donnee
SLOPE_RECLASS_LUT = np.full(11, -1, dtype=np.int8)
for code10, label8 in SLOPEMED_10_TO_A68_8.items():
    SLOPE_RECLASS_LUT[code10] = SLOPE_LABEL_TO_IDX[label8]

print('LUT reclassification pente (brut -> indice A6-8, -1=pas de donnee):', SLOPE_RECLASS_LUT.tolist())


LUT reclassification pente (brut -> indice A6-8, -1=pas de donnee): [-1, 0, 1, 2, 3, 4, 4, 5, 5, 6, 7]


## 4. Table de notation (S1/S2/N -> echelle ordinale)

On convertit les classes litterales en echelle ordinale 0 (N, non adapte) a 4 (S1, optimal) pour
pouvoir la stocker/agreger numeriquement (moyenne par bloc, meme logique que SSR pour le sol).

In [4]:
RATING_TO_ORDINAL = {'N': 0, 'S2/N': 1, 'S2': 2, 'S1/N': 2, 'S1/S2': 3, 'S1': 4}

# NB : la table source A6-8 ne redonne la note des pentes <5% (0-0.5/0.5-2/2-5) que sous FM<1300 
# (l'intensite des pluies ne joue pas de role erosif a ces pentes quasi-nulles) ; 
# ces valeurs ont ete reportees (fill-forward) sur les 5 autres classes de Fournier lors du parsing.
terrain_df = pd.read_csv(TERRAIN_CSV)
terrain_df['ordinal'] = terrain_df['rating'].map(RATING_TO_ORDINAL)
assert terrain_df['ordinal'].isna().sum() == 0, "valeurs de rating non reconnues"

# dict rapide : (water_system, crop_group, input_level, fm_label, slope_label) -> ordinal
RATING_LUT = {}
for row in terrain_df.itertuples():
    RATING_LUT[(row.water_system, row.crop_group, row.input_level, row.fm_class, row.slope_class)] = row.ordinal

print('table chargee :', len(terrain_df), 'lignes ;', len(RATING_LUT), 'combinaisons uniques')


def build_crop_rating_array(water_system, crop_group, input_level):
    """Retourne un tableau (6 classes FM, 8 classes pente) d'ordinaux, pour lookup vectorise rapide."""
    arr = np.full((6, 8), -1, dtype=np.int8)
    for fi, fm_label in enumerate(FM_LABELS):
        for si, slope_label in enumerate(SLOPE_CLASSES_A68):
            key = (water_system, crop_group, input_level, fm_label, slope_label)
            if key in RATING_LUT:
                arr[fi, si] = RATING_LUT[key]
    assert (arr >= 0).all(), f"combinaison manquante pour {water_system}/{crop_group}/{input_level}"
    return arr


table chargee : 2592 lignes ; 2592 combinaisons uniques


## 5. Calcul pour une source de precipitation donnee

Pour chaque pixel natif de pente (30 arc-sec), on determine sa classe de pente (section 3), on
recupere la classe de Fournier de la cellule climatique dont il depend (plus proche voisin — downscaling
par repetition, puisque le climat est a resolution bien plus grossiere que la pente), puis on fait le
lookup (culture x regime x niveau) -> note ordinale. Resultat agrege ensuite par bloc (moyenne simple,
meme logique que `2.e)` pour le sol) a la resolution de sortie voulue.

In [5]:
def compute_terrain_rating_native(fm_class_da, out_agg_factor, window_bounds=None):
    """fm_class_da : DataArray 2D (classe de Fournier 0-5) sur la grille climat (lat/lon en degres).
    Retourne, pour chaque (water_system, crop_group, input_level) unique necessaire, un dict de
    rasters agreges (ordinal moyen par bloc de out_agg_factor pixels natifs)."""
    lat_c = fm_class_da['lat'].values
    lon_c = fm_class_da['lon'].values
    fm_vals = fm_class_da.values  # (nlat, nlon), int, -1 si NaN/manquant

    with rasterio.open(SLOPE_TIF) as src:
        if window_bounds is None:
            # etendue dynamique = celle du fichier climat (pas de zone codee en dur)
            lat_min, lat_max = float(lat_c.min()), float(lat_c.max())
            lon_min, lon_max = float(lon_c.min()), float(lon_c.max())
        else:
            lat_min, lat_max, lon_min, lon_max = window_bounds
        r0, c0 = rowcol(src.transform, lon_min, lat_max)
        r1, c1 = rowcol(src.transform, lon_max, lat_min)
        r0, r1 = max(0, r0), min(src.height, r1)
        c0, c1 = max(0, c0), min(src.width, c1)
        # arrondi aux multiples de out_agg_factor pour un reshape propre
        h = ((r1 - r0) // out_agg_factor) * out_agg_factor
        w = ((c1 - c0) // out_agg_factor) * out_agg_factor
        slope_raw = src.read(1, window=Window(c0, r0, w, h))
        slope_idx = SLOPE_RECLASS_LUT[slope_raw]  # (h, w), -1 si eau/pas de donnee

        # coordonnees natives de chaque pixel -> indices dans la grille climat (plus proche voisin)
        native_lats = np.array([src.transform * (0, rr) for rr in range(r0, r0 + h)])[:, 1]
        native_lons = np.array([src.transform * (cc, 0) for cc in range(c0, c0 + w)])[:, 0]
    lat_idx = np.abs(lat_c[:, None] - native_lats[None, :]).argmin(axis=0)  # (h,)
    lon_idx = np.abs(lon_c[:, None] - native_lons[None, :]).argmin(axis=0)  # (w,)
    fm_native = fm_vals[lat_idx][:, lon_idx]  # (h, w) classe FM par pixel natif

    valid = (slope_idx >= 0) & (fm_native >= 0)
    return slope_idx, fm_native, valid, h, w, r0, c0


def aggregate_ordinal(rating_native, valid, agg_factor):
    out_h, out_w = rating_native.shape[0] // agg_factor, rating_native.shape[1] // agg_factor
    vals = np.where(valid, rating_native, np.nan).astype(np.float32)
    vals_r = vals.reshape(out_h, agg_factor, out_w, agg_factor)
    with np.errstate(invalid='ignore'):
        return np.nanmean(vals_r, axis=(1, 3))


## 6. Boucle principale — ERA5 (reference) + scenarios AMOC

In [6]:
def load_daily_pr(path, var='pr'):
    ds = xr.open_dataset(path)
    da = ds[var]
    if 'lon' in da.dims and 'lat' in da.dims:
        da = da.transpose('time', 'lat', 'lon')
    return da.load()  # charge en memoire une fois (evite des centaines de lectures disque lazy, une par groupe resample/groupby)


def save_terrain_outputs(ordinal_agg, lat_out, lon_out, source_label, crop_label, regime, tag_out):
    da = xr.DataArray(ordinal_agg, dims=('lat', 'lon'), coords={'lat': lat_out, 'lon': lon_out},
                       name=f'terrain_rating_{crop_label}_{regime}',
                       attrs={'source': 'GAEZ v4 Appendix 6-8 (Table A6-8.1/A6-8.2) x SLOPE-MED x '
                                        f'Fournier({source_label})',
                              'legend': 'ordinal 0=N(unsuitable) 1=S2/N 2=S2 3=S1/S2 4=S1(optimal)',
                              'crop': crop_label, 'regime': regime, 'input_level': INPUT_LEVEL,
                              'climate_source': source_label})
    fn = OUT_DIR / f'terrain_rating_{crop_label}_{regime}_{tag_out}.nc'
    da.to_netcdf(fn)
    return fn


results_log = []

# --- A) ERA5 (reference), sortie a 0.25deg (natif) et 1deg (agregation supplementaire) ---
print('=== ERA5 (reference) ===')
pr_era5 = load_daily_pr(ERA5_PR)
fm_era5 = compute_fournier(pr_era5)
fm_class_era5 = classify_fournier(fm_era5)

for agg_factor, tag_out in [(30, '0.25deg'), (120, '1deg')]:
    slope_idx, fm_native, valid, h, w, r0, c0 = compute_terrain_rating_native(fm_class_era5, agg_factor)
    lat_out = np.array([90 - (r0 + i * agg_factor + agg_factor / 2) / 120 for i in range(h // agg_factor)])
    lon_out = np.array([-180 + (c0 + j * agg_factor + agg_factor / 2) / 120 for j in range(w // agg_factor)])
    for crop in CROPS:
        crop_group = CROP_TO_GROUP[crop]
        for regime, water_system in REGIME_TO_WATER_SYSTEM.items():
            rating_table = build_crop_rating_array(water_system, crop_group, INPUT_LEVEL)
            rating_native = np.where(valid, rating_table[fm_native.clip(0), slope_idx.clip(0)], np.nan)
            agg = aggregate_ordinal(rating_native, valid, agg_factor)
            for cname in CROP_NAME_ALIASES[crop]:
                fn = save_terrain_outputs(agg, lat_out, lon_out, 'ERA5', cname, regime, f'ERA5_{tag_out}')
    print('  ', tag_out, '-> sauvegarde pour', len(CROPS), 'cultures x 2 regimes')
    results_log.append(('ERA5', tag_out, 'OK'))

# --- B) Scenarios AMOC (EC-Earth direct + 20CR-solution), sortie a 1deg (natif de ces sources) ---
for source_label, src_cfg in AMOC_SOURCES.items():
    for tag in SCENARIO_TAGS:
        f = src_cfg['dir'] / src_cfg['pattern'].format(tag=tag)
        if not f.exists():
            print(f'  SKIP {source_label} / {tag} -> fichier absent ({f.name}), pas encore livre')
            results_log.append((source_label, tag, 'MISSING'))
            continue
        print(f'=== {source_label} / {tag} ===')
        pr_scn = load_daily_pr(f)
        fm_scn = compute_fournier(pr_scn)
        fm_class_scn = classify_fournier(fm_scn)
        agg_factor = 120  # ces sources sont deja nativement ~1deg -> on vise directement 1deg
        slope_idx, fm_native, valid, h, w, r0, c0 = compute_terrain_rating_native(fm_class_scn, agg_factor)
        lat_out = np.array([90 - (r0 + i * agg_factor + agg_factor / 2) / 120 for i in range(h // agg_factor)])
        lon_out = np.array([-180 + (c0 + j * agg_factor + agg_factor / 2) / 120 for j in range(w // agg_factor)])
        for crop in CROPS:
            crop_group = CROP_TO_GROUP[crop]
            for regime, water_system in REGIME_TO_WATER_SYSTEM.items():
                rating_table = build_crop_rating_array(water_system, crop_group, INPUT_LEVEL)
                rating_native = np.where(valid, rating_table[fm_native.clip(0), slope_idx.clip(0)], np.nan)
                agg = aggregate_ordinal(rating_native, valid, agg_factor)
                for cname in CROP_NAME_ALIASES[crop]:
                    save_terrain_outputs(agg, lat_out, lon_out, f'{source_label}_{tag}', cname, regime,
                                          f'{source_label}_{tag}_1deg')
        print('  -> sauvegarde pour', len(CROPS), 'cultures x 2 regimes')
        results_log.append((source_label, tag, 'OK'))

print()
print('=== Bilan ===')
for r in results_log:
    print('  ', r)


=== ERA5 (reference) ===


C:\Users\elisa\AppData\Local\Temp\ipykernel_38484\3131277012.py:42: RuntimeWarning: Mean of empty slice
  return np.nanmean(vals_r, axis=(1, 3))


   0.25deg -> sauvegarde pour 5 cultures x 2 regimes


   1deg -> sauvegarde pour 5 cultures x 2 regimes
=== ECEarth / 2025_hosing ===


C:\Users\elisa\AppData\Local\Temp\ipykernel_38484\3131277012.py:42: RuntimeWarning: Mean of empty slice
  return np.nanmean(vals_r, axis=(1, 3))


...

  -> sauvegarde pour 5 cultures x 2 regimes
  SKIP ECEarth / 2050_hosing -> fichier absent (pr_adj_2050_hosing.nc), pas encore livre
  SKIP ECEarth / 2050_nohosing -> fichier absent (pr_adj_2050_nohosing.nc), pas encore livre
=== 20CRsolution / 2025_hosing ===


C:\Users\elisa\AppData\Local\Temp\ipykernel_38484\3131277012.py:42: RuntimeWarning: Mean of empty slice
  return np.nanmean(vals_r, axis=(1, 3))


  -> sauvegarde pour 5 cultures x 2 regimes
=== 20CRsolution / 2025_nohosing ===


C:\Users\elisa\AppData\Local\Temp\ipykernel_38484\3131277012.py:42: RuntimeWarning: Mean of empty slice
  return np.nanmean(vals_r, axis=(1, 3))


  -> sauvegarde pour 5 cultures x 2 regimes
  SKIP 20CRsolution / 2050_hosing -> fichier absent (pr_adj_2050_hosing_combined.nc), pas encore livre
  SKIP 20CRsolution / 2050_nohosing -> fichier absent (pr_adj_2050_nohosing_combined.nc), pas encore livre

=== Bilan ===
   ('ERA5', '0.25deg', 'OK')
   ('ERA5', '1deg', 'OK')
   ('ECEarth', '2025_hosing', 'OK')
   ('ECEarth', '2025_nohosing', 'OK')
   ('ECEarth', '2050_hosing', 'MISSING')
   ('ECEarth', '2050_nohosing', 'MISSING')
   ('20CRsolution', '2025_hosing', 'OK')
   ('20CRsolution', '2025_nohosing', 'OK')
   ('20CRsolution', '2050_hosing', 'MISSING')
   ('20CRsolution', '2050_nohosing', 'MISSING')


## 7. Verification rapide

Sanity-check : zone plate (devrait donner S1/4 presque partout) vs zone tres pentue (devrait montrer
des notes degradees), sur la sortie ERA5/0.25deg/Wheat/rf.

In [7]:
check = xr.open_dataarray(OUT_DIR / 'terrain_rating_Wheat-spring_rf_ERA5_0.25deg.nc')
print('Pays-Bas (plat), note terrain Wheat/rf (attendu proche de 4) :',
      float(check.sel(lat=52.25, lon=5.5, method='nearest').values))
print('Alpes suisses (pentu), note terrain Wheat/rf (attendu nettement < 4) :',
      float(check.sel(lat=46.5, lon=8.0, method='nearest').values))


Pays-Bas (plat), note terrain Wheat/rf (attendu proche de 4) : 4.0
Alpes suisses (pentu), note terrain Wheat/rf (attendu nettement < 4) : 0.3722222149372101


## 8. Suite

- Ce notebook produit la note terrain-pente (ordinale 0-4) par pixel, culture, regime, et source
  climatique (ERA5 + scenarios AMOC disponibles) — prete a etre utilisee au Module V pour degrader la
  classe de suitability finale (climat+sol), comme documente dans
  `0_reference/appendix_tables/Appendix_6-8_Terrain_Slope_Ratings_GAEZ_V4.pdf`.
- Quand les fichiers 2050_hosing/2050_nohosing seront disponibles (meme emplacement, meme format que
  2025), il suffira de relancer ce notebook — la boucle les detectera automatiquement.
- Quand les fichiers AMOC passeront a une couverture mondiale, aucune modification de code n'est
  necessaire — l'etendue spatiale est lue dynamiquement depuis chaque fichier d'entree.
